In [4]:
# example notebook to run in QuantBook cloud environment
from QuantConnect import *
from QuantConnect.Research import QuantBook
from datetime import datetime, timedelta
from QuantConnect.Data.Market import OptionRight, OptionUniverse

ModuleNotFoundError: No module named 'QuantConnect'

In [2]:
qb = QuantBook()
index = qb.add_index("SPX")

NameError: name 'QuantBook' is not defined

In [ ]:
option = qb.add_index_option("SPX", "SPXW")
option.set_filter(lambda u: u.expiration(timedelta(0), timedelta(7)).strikes(-15, 15))
future = qb.add_future("MES", Resolution.DAILY)
history = qb.history[OptionUniverse](option.symbol, datetime(2026, 9, 7), datetime(2026, 9, 12))
print(f"Chain snapshots: {len(history)}")
if history:
    latest = list(history)[-1]
    print(f"Latest chain date: {latest.end_time}")
    print(f"Contracts: {len(list(latest))}")

In [ ]:
import pandas as pd
import numpy as np

# Spot: SPX close on the latest chain date
spx_history = qb.history(index.symbol, datetime(2026, 9, 8), datetime(2026, 9, 12), Resolution.DAILY)
spx_close = float(spx_history["close"].iloc[-1])
spx_date = spx_history.index.get_level_values("time")[-1]
print(f"SPX close {spx_close:.2f} on {spx_date.date()}")

# Filter to near-dated expiries and a reasonable strike band around spot
contracts = [c for c in latest
             if c.implied_volatility > 0 and c.greeks.gamma and c.open_interest > 0
             and c.symbol.id.date.date() <= (latest.end_time + timedelta(days=7)).date()
             and abs(c.symbol.id.strike_price - spx_close) < 300]
print(f"Contracts used for GEX: {len(contracts)}")

# GEX per contract: gamma * OI * 100 * spot^2 * 0.01 -> dollar gamma per 1% move
# Dealer convention: calls +, puts -
rows = []
for c in contracts:
    strike = c.symbol.id.strike_price
    expiry = c.symbol.id.date
    right = c.symbol.id.option_right
    gex = c.greeks.gamma * c.open_interest * 100 * spx_close**2 * 0.01
    sign = 1 if right == OptionRight.CALL else -1
    dte = (expiry - latest.end_time).days
    rows.append({"strike": strike, "expiry": expiry, "dte": dte, "right": str(right),
                 "gex": sign * gex, "oi": c.open_interest, "gamma": c.greeks.gamma})
df = pd.DataFrame(rows)

net_gex = df["gex"].sum() / 1e9
print(f"Net GEX (0-7 DTE, +/-300 pts): {net_gex:.2f} $bn per 1% move")
by_strike = df.groupby("strike")["gex"].sum().sort_index()
top_walls = by_strike.abs().sort_values(ascending=False).head(5)
print("Top gamma walls:")
for k in top_walls.index:
    print(f"  {k}: {by_strike[k]/1e9:+.2f} $bn")

In [ ]:
# Zero-gamma flip point: cumulative net GEX walking up from low strikes
cum = by_strike.cumsum()
flips = cum[cum.shift(1).fillna(0) * cum < 0]
if len(flips):
    flip = flips.index[0]
else:
    flip = by_strike.index[0] if cum.iloc[0] > 0 else by_strike.index[-1]
print(f"Zero-gamma flip point: {flip}")

# GEX by expiry bucket (0DTE vs rest)
by_dte = df.groupby(["dte"])["gex"].sum().sort_index() / 1e9
print("Net GEX by DTE:")
print(by_dte.round(2).to_string())

In [ ]:
# Monday's effective profile: drop Friday's expired 0DTE contracts
df_live = df[df.dte > 0]
by_strike_live = df_live.groupby("strike")["gex"].sum().sort_index()
cum_live = by_strike_live.cumsum()
flips_live = cum_live[cum_live.shift(1).fillna(0) * cum_live < 0]
flip_live = flips_live.index[0] if len(flips_live) else None
net_live = df_live["gex"].sum() / 1e9
print(f"Net GEX (Monday, 3-7 DTE): {net_live:.2f} $bn")
print(f"Flip point (Monday): {flip_live}")
walls_live = by_strike_live.abs().sort_values(ascending=False).head(5)
print("Top walls (Monday):")
for k in walls_live.index:
    print(f"  {k}: {by_strike_live[k]/1e9:+.2f} $bn")
put_wall = by_strike_live[by_strike_live < 0].abs().idxmax()
call_wall = by_strike_live[by_strike_live > 0].abs().idxmax() if (by_strike_live > 0).any() else None
print(f"Primary put wall: {put_wall}, primary call wall: {call_wall}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

mes = qb.history(future.symbol, datetime(2026, 8, 28), datetime(2026, 9, 12), Resolution.DAILY)
closes = mes["close"].droplevel(["expiry", "symbol"])
closes.index = closes.index.normalize()
print(f"MES sessions: {len(closes)}, last close {closes.iloc[-1]:.2f} on {closes.index[-1].date()}")

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(closes.index, closes.values, color="black", lw=1.6, label="MES (continuous)")
levels = [(7500, "Put wall", "red"), (7550, "Put wall 2", "salmon"),
          (7700, "Call wall", "green"), (flip_live, "Flip point", "purple") if flip_live else (7890, "Flip point (w/ 0DTE)", "purple")]
for lvl, label, color in levels:
    ax.axhline(lvl, color=color, ls="--", lw=1.2)
    ax.text(closes.index[0], lvl + 6, f"{label} {lvl}", color=color, fontsize=9)
ax.axhline(spx_close, color="blue", ls=":", lw=1.2)
ax.text(closes.index[0], spx_close + 6, f"SPX spot {spx_close:.0f}", color="blue", fontsize=9)
ax.set_title("MES Pre-Market Prep - SPX GEX Levels (data: 2026-09-11 close)")
ax.set_ylabel("Index points")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax.grid(alpha=0.3)
plt.show()

# Pre-Market Prep Sheet - Monday 2026-09-14

Data as of Friday 2026-09-11 close (prior EOD chain; correct for pre-market prep).

| Item | Value |
|---|---|
| SPX close | 7,592.30 (MES last: 7,660.75, futures basis ~ +68 pts) |
| Net GEX (3-7 DTE) | -$15.0bn per 1% move - NEGATIVE gamma regime |
| Zero-gamma flip | None within +/-300 pts (deeply short gamma) |
| Primary put wall | 7,500 (-$1.6bn) |
| Secondary put walls | 7,550 (-$1.3bn), 7,625 (-$1.3bn), 7,600 (-$1.0bn) |
| Call wall | 7,700 (+) - capped upside |